In [103]:
!pip install pandas requests

In [104]:
import requests
import pandas as pd
from io import StringIO

In [105]:
query = """
SELECT
    pl_name,
    hostname,
    pl_masse,
    pl_rade,
    pl_orbper,
    pl_orbsmax,
    st_mass,
    st_rad,
    st_teff,
    sy_snum,
    sy_pnum,
    disc_year,
    discoverymethod,
    disc_facility,
    disc_telescope,
    disc_instrument
FROM pscomppars
"""

In [106]:
url = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync"

params = {
    "query": query,
    "format": "csv"
}

respuesta = requests.get(url, params=params)

In [107]:
respuesta.status_code

200

In [108]:
df = pd.read_csv(
    StringIO(respuesta.text)
)

In [109]:
df.head()

,pl_name,hostname,pl_masse,pl_rade,pl_orbper,pl_orbsmax,st_mass,st_rad,st_teff,sy_snum,sy_pnum,disc_year,discoverymethod,disc_facility,disc_telescope,disc_instrument
0,HD 2039 b,HD 2039,NaN,12.700000,1120.000000,2.20000,1.230,1.190,5945.0,1,1,2002,Radial Velocity,Anglo-Australian Telescope,3.9 m Anglo-Australian Telescope,UCLES Spectrograph
1,HAT-P-8 b,HAT-P-8,406.8224,15.692600,3.076340,0.04496,1.270,1.570,6200.0,3,1,2008,Transit,HATNet,Canon 200mm f/1.8L,2K CCD Sensor
2,K2-43 b,K2-43,NaN,4.510000,3.471149,0.03784,0.571,0.542,3840.6,1,2,2016,Transit,K2,0.95 m Kepler Telescope,Kepler CCD Array
3,Kepler-1753 b,Kepler-1753,NaN,2.226781,16.004601,0.11790,0.860,0.769,5180.0,1,1,2021,Transit,Kepler,0.95 m Kepler Telescope,Kepler CCD Array
4,Kepler-1176 b,Kepler-1176,NaN,2.450000,24.173858,0.15890,1.010,1.020,5844.0,1,1,2016,Transit,Kepler,0.95 m Kepler Telescope,Kepler CCD Array


In [110]:
df.shape

(6360, 16)

In [111]:
len(df)

6360

In [112]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6360 entries, 0 to 6359
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   pl_name          6360 non-null   object 
 1   hostname         6360 non-null   object 
 2   pl_masse         2448 non-null   float64
 3   pl_rade          6310 non-null   float64
 4   pl_orbper        6014 non-null   float64
 5   pl_orbsmax       5930 non-null   float64
 6   st_mass          6351 non-null   float64
 7   st_rad           6036 non-null   float64
 8   st_teff          6060 non-null   float64
 9   sy_snum          6360 non-null   int64  
 10  sy_pnum          6360 non-null   int64  
 11  disc_year        6360 non-null   int64  
 12  discoverymethod  6360 non-null   object 
 13  disc_facility    6360 non-null   object 
 14  disc_telescope   6360 non-null   object 
 15  disc_instrument  6360 non-null   object 
dtypes: float64(7), int64(3), object(6)
memory usage: 795.1+ KB


In [113]:
df.to_csv(
    "exoplanetas_nasa.csv",
    index=False
)

In [114]:
df.isna().sum()

,0
pl_name,0
hostname,0
pl_masse,3912
pl_rade,50
pl_orbper,346
pl_orbsmax,430
st_mass,9
st_rad,324
st_teff,300
sy_snum,0


In [115]:
df.isna().sum().sort_values(ascending=False)

,0
pl_masse,3912
pl_orbsmax,430
pl_orbper,346
st_rad,324
st_teff,300
pl_rade,50
st_mass,9
pl_name,0
hostname,0
sy_snum,0


In [116]:
df["hostname"].nunique()

4769

In [117]:
df["hostname"].value_counts().head(10)

,count
hostname,
KOI-351,8
TRAPPIST-1,7
K2-138,6
Kepler-20,6
HIP 41378,6
HD 34445,6
HD 219134,6
Kepler-80,6
HD 110067,6


In [118]:
estrellas = df[
    [
        "hostname",
        "st_mass",
        "st_rad",
        "st_teff"
    ]
].copy()

In [119]:
estrellas["hostname"].nunique()

4769

In [120]:
len(estrellas)

6360

In [121]:
estrellas = estrellas.drop_duplicates(
    subset=["hostname"]
)

In [122]:
len(estrellas)

4769

In [123]:
estrellas["hostname"].nunique()

4769

In [124]:
estrellas = estrellas.reset_index(drop=True)

estrellas.insert(
    0,
    "id_estrella",
    range(1, len(estrellas) + 1)
)

In [125]:
estrellas = estrellas.rename(columns={
    "hostname": "nombre",
    "st_mass": "masa",
    "st_rad": "radio",
    "st_teff": "temperatura"
})

In [126]:
planetas = df[
    [
        "pl_name",
        "hostname",
        "pl_masse",
        "pl_rade",
        "pl_orbper",
        "pl_orbsmax"
    ]
].copy()

In [127]:
planetas = planetas.rename(columns={
    "pl_name": "nombre",
    "pl_masse": "masa",
    "pl_rade": "radio",
    "pl_orbper": "periodo_orbital",
    "pl_orbsmax": "semieje_mayor"
})

In [128]:
planetas = planetas.merge(
    estrellas[["id_estrella", "nombre"]],
    left_on="hostname",
    right_on="nombre",
    how="left"
)

In [129]:
planetas.head()

,nombre_x,hostname,masa,radio,periodo_orbital,semieje_mayor,id_estrella,nombre_y
0,HD 2039 b,HD 2039,NaN,12.700000,1120.000000,2.20000,1,HD 2039
1,HAT-P-8 b,HAT-P-8,406.8224,15.692600,3.076340,0.04496,2,HAT-P-8
2,K2-43 b,K2-43,NaN,4.510000,3.471149,0.03784,3,K2-43
3,Kepler-1753 b,Kepler-1753,NaN,2.226781,16.004601,0.11790,4,Kepler-1753
4,Kepler-1176 b,Kepler-1176,NaN,2.450000,24.173858,0.15890,5,Kepler-1176


In [130]:
planetas = planetas.drop(
    columns=["nombre_y"]
)

In [131]:
planetas = planetas.rename(
    columns={"nombre_x": "nombre"}
)

In [132]:
planetas.head()

,nombre,hostname,masa,radio,periodo_orbital,semieje_mayor,id_estrella
0,HD 2039 b,HD 2039,NaN,12.700000,1120.000000,2.20000,1
1,HAT-P-8 b,HAT-P-8,406.8224,15.692600,3.076340,0.04496,2
2,K2-43 b,K2-43,NaN,4.510000,3.471149,0.03784,3
3,Kepler-1753 b,Kepler-1753,NaN,2.226781,16.004601,0.11790,4
4,Kepler-1176 b,Kepler-1176,NaN,2.450000,24.173858,0.15890,5


In [133]:
planetas["id_estrella"].isna().sum()

np.int64(0)

In [134]:
planetas = planetas.reset_index(drop=True)

planetas.insert(
    0,
    "id_planeta",
    range(1, len(planetas) + 1)
)

In [135]:
planetas = planetas.drop(
    columns=["hostname"]
)

In [136]:
planetas.head()

,id_planeta,nombre,masa,radio,periodo_orbital,semieje_mayor,id_estrella
0,1,HD 2039 b,NaN,12.700000,1120.000000,2.20000,1
1,2,HAT-P-8 b,406.8224,15.692600,3.076340,0.04496,2
2,3,K2-43 b,NaN,4.510000,3.471149,0.03784,3
3,4,Kepler-1753 b,NaN,2.226781,16.004601,0.11790,4
4,5,Kepler-1176 b,NaN,2.450000,24.173858,0.15890,5


PLANETA
id_planeta
id_estrella     ← FK
nombre
masa
radio
periodo_orbital
semieje_mayor

In [137]:
descubrimientos = df[
    [
        "pl_name",
        "disc_year",
        "discoverymethod",
        "disc_facility",
        "disc_telescope",
        "disc_instrument"
    ]
].copy()

In [138]:
descubrimientos = descubrimientos.rename(columns={
    "pl_name": "planeta",
    "disc_year": "año",
    "discoverymethod": "metodo",
    "disc_facility": "instalacion",
    "disc_telescope": "telescopio",
    "disc_instrument": "instrumento"
})

In [139]:
descubrimientos = descubrimientos.merge(
    planetas[["id_planeta", "nombre"]],
    left_on="planeta",
    right_on="nombre",
    how="left"
)

In [140]:
descubrimientos.head()

,planeta,año,metodo,instalacion,telescopio,instrumento,id_planeta,nombre
0,HD 2039 b,2002,Radial Velocity,Anglo-Australian Telescope,3.9 m Anglo-Australian Telescope,UCLES Spectrograph,1,HD 2039 b
1,HAT-P-8 b,2008,Transit,HATNet,Canon 200mm f/1.8L,2K CCD Sensor,2,HAT-P-8 b
2,K2-43 b,2016,Transit,K2,0.95 m Kepler Telescope,Kepler CCD Array,3,K2-43 b
3,Kepler-1753 b,2021,Transit,Kepler,0.95 m Kepler Telescope,Kepler CCD Array,4,Kepler-1753 b
4,Kepler-1176 b,2016,Transit,Kepler,0.95 m Kepler Telescope,Kepler CCD Array,5,Kepler-1176 b


In [141]:
descubrimientos = descubrimientos.drop(
    columns=["nombre"]
)

In [143]:
descubrimientos = descubrimientos.rename(
    columns={"planeta": "nombre_planeta"}
)

In [144]:
descubrimientos.head()

,nombre_planeta,año,metodo,instalacion,telescopio,instrumento,id_planeta
0,HD 2039 b,2002,Radial Velocity,Anglo-Australian Telescope,3.9 m Anglo-Australian Telescope,UCLES Spectrograph,1
1,HAT-P-8 b,2008,Transit,HATNet,Canon 200mm f/1.8L,2K CCD Sensor,2
2,K2-43 b,2016,Transit,K2,0.95 m Kepler Telescope,Kepler CCD Array,3
3,Kepler-1753 b,2021,Transit,Kepler,0.95 m Kepler Telescope,Kepler CCD Array,4
4,Kepler-1176 b,2016,Transit,Kepler,0.95 m Kepler Telescope,Kepler CCD Array,5


In [145]:
telescopios = df[
    [
        "disc_facility",
        "disc_telescope",
        "disc_instrument"
    ]
].drop_duplicates()

In [146]:
telescopios = telescopios.rename(columns={
    "disc_facility": "instalacion",
    "disc_telescope": "nombre",
    "disc_instrument": "instrumento"
})

In [147]:
telescopios = telescopios.reset_index(drop=True)

telescopios.insert(
    0,
    "id_telescopio",
    range(1, len(telescopios) + 1)
)

In [151]:
telescopios.head()

,id_telescopio,instalacion,nombre,instrumento
0,1,Anglo-Australian Telescope,3.9 m Anglo-Australian Telescope,UCLES Spectrograph
1,2,HATNet,Canon 200mm f/1.8L,2K CCD Sensor
2,3,K2,0.95 m Kepler Telescope,Kepler CCD Array
3,4,Kepler,0.95 m Kepler Telescope,Kepler CCD Array
4,5,SuperWASP,Canon 200mm f/1.8L,iKon-L CCD Camera


In [148]:
descubrimientos = descubrimientos.merge(
    telescopios[
        [
            "id_telescopio",
            "nombre",
            "instalacion",
            "instrumento"
        ]
    ],
    left_on=[
        "telescopio",
        "instalacion",
        "instrumento"
    ],
    right_on=[
        "nombre",
        "instalacion",
        "instrumento"
    ],
    how="left"
)

In [149]:
descubrimientos = descubrimientos.reset_index(drop=True)

descubrimientos.insert(
    0,
    "id_descubrimiento",
    range(1, len(descubrimientos) + 1)
)

In [150]:
descubrimientos = descubrimientos[
    [
        "id_descubrimiento",
        "id_planeta",
        "id_telescopio",
        "año",
        "metodo"
    ]
]

In [152]:
descubrimientos.head()

,id_descubrimiento,id_planeta,id_telescopio,año,metodo
0,1,1,1,2002,Radial Velocity
1,2,2,2,2008,Transit
2,3,3,3,2016,Transit
3,4,4,4,2021,Transit
4,5,5,4,2016,Transit


In [153]:
estrellas.to_csv(
    "estrella.csv",
    index=False
)

planetas.to_csv(
    "planeta.csv",
    index=False
)

telescopios.to_csv(
    "telescopio.csv",
    index=False
)

descubrimientos.to_csv(
    "descubrimiento.csv",
    index=False
)

![Diagrama del modelo relacional a construir](diagrama_relacional.png)

# **CEAR LA BASE DE DATOS RELACIONAL**

## 1. Creación e inspección de la base de datos

```sql
CREATE DATABASE astronomia;

USE astronomia;

SHOW DATABASES;

SELECT DATABASE();

```

---

## 2. Tabla `estrella`

```sql
CREATE TABLE estrella (
    id_estrella INT AUTO_INCREMENT,
    nombre VARCHAR(100) NOT NULL,
    masa DECIMAL(10,4),
    radio DECIMAL(10,4),
    temperatura INT,

    PRIMARY KEY (id_estrella),

    UNIQUE (nombre)
);

```

### Descripción de campos:

* **`id_estrella`** → Clave Primaria (PK)
* **`nombre`** → Nombre de la estrella
* **`masa`** → Masas solares ($M_\odot$)
* **`radio`** → Radios solares ($R_\odot$)
* **`temperatura`** → Temperatura efectiva ($K$)

```sql
DESCRIBE estrella;

```

---

## 3. Tabla `planeta`

```sql
CREATE TABLE planeta (
    id_planeta INT AUTO_INCREMENT,
    id_estrella INT NOT NULL,
    nombre VARCHAR(100) NOT NULL,
    masa DECIMAL(12,5),
    radio DECIMAL(10,5),
    periodo_orbital DECIMAL(15,6),
    semieje_mayor DECIMAL(15,6),

    PRIMARY KEY (id_planeta),

    UNIQUE (nombre),

    FOREIGN KEY (id_estrella)
        REFERENCES estrella(id_estrella)
);

```

### Claves foráneas:

* `FOREIGN KEY (id_estrella) REFERENCES estrella(id_estrella)`

---

## 4. Tabla `telescopio`

```sql
CREATE TABLE telescopio (
    id_telescopio INT AUTO_INCREMENT,
    nombre VARCHAR(150),
    instalacion VARCHAR(150),
    instrumento VARCHAR(150),

    PRIMARY KEY (id_telescopio)
);

```

```sql
DESCRIBE telescopio;

```

---

## 5. Tabla `descubrimiento`

```sql
CREATE TABLE descubrimiento (
    id_descubrimiento INT AUTO_INCREMENT,
    id_planeta INT NOT NULL,
    id_telescopio INT,
    año INT,
    metodo VARCHAR(100),

    PRIMARY KEY (id_descubrimiento),

    FOREIGN KEY (id_planeta)
        REFERENCES planeta(id_planeta),

    FOREIGN KEY (id_telescopio)
        REFERENCES telescopio(id_telescopio)
);

```

---

## 6. Verificación final

```sql
SHOW TABLES;

```

# CREACIÓN DE LA BASE DE DATOS Y TABLAS DESDE PYTHON

Podemos automatizar la creación de la base de datos y su estructura ejecutando comandos DDL directamente desde un script de Python usando la librería `mysql-connector-python`.

---

## 1. Instalación de la librería

Si no tienes instalada la librería en tu entorno, ejecuta primero:

```bash
!pip install mysql-connector-python


In [ ]:
!pip install mysql-connector-python

## 2. Script para la creaciónde DB.


In [ ]:
import mysql.connector
from mysql.connector import Error

# 1. Configuración de parámetros de conexión
config = {
    'host': 'localhost',
    'user': 'root',
    'password': 'tu_contraseña'  # Reemplazar con tu contraseña de MySQL
}

try:
    # 2. Conexión al servidor MySQL
    conexion = mysql.connector.connect(**config)
    cursor = conexion.cursor()

    # 3. Crear la base de datos 'astronomia'
    cursor.execute("CREATE DATABASE IF NOT EXISTS astronomia;")
    cursor.execute("USE astronomia;")
    print("Base de datos 'astronomia' lista.")

    # 4. Crear tabla ESTRELLA (Padre)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS estrella (
        id_estrella INT AUTO_INCREMENT,
        nombre VARCHAR(100) NOT NULL,
        masa DECIMAL(10,4),
        radio DECIMAL(10,4),
        temperatura INT,
        PRIMARY KEY (id_estrella),
        UNIQUE (nombre)
    );
    """)

    # 5. Crear tabla TELESCOPIO (Padre)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS telescopio (
        id_telescopio INT AUTO_INCREMENT,
        nombre VARCHAR(150),
        instalacion VARCHAR(150),
        instrumento VARCHAR(150),
        PRIMARY KEY (id_telescopio)
    );
    """)

    # 6. Crear tabla PLANETA (Hijo de estrella)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS planeta (
        id_planeta INT AUTO_INCREMENT,
        id_estrella INT NOT NULL,
        nombre VARCHAR(100) NOT NULL,
        masa DECIMAL(12,5),
        radio DECIMAL(10,5),
        periodo_orbital DECIMAL(15,6),
        semieje_mayor DECIMAL(15,6),
        PRIMARY KEY (id_planeta),
        UNIQUE (nombre),
        FOREIGN KEY (id_estrella) REFERENCES estrella(id_estrella)
    );
    """)

    # 7. Crear tabla DESCUBRIMIENTO (Hijo de planeta y telescopio)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS descubrimiento (
        id_descubrimiento INT AUTO_INCREMENT,
        id_planeta INT NOT NULL,
        id_telescopio INT,
        año INT,
        metodo VARCHAR(100),
        PRIMARY KEY (id_descubrimiento),
        FOREIGN KEY (id_planeta) REFERENCES planeta(id_planeta),
        FOREIGN KEY (id_telescopio) REFERENCES telescopio(id_telescopio)
    );
    """)

    print("Todas las tablas fueron creadas exitosamente respetando la integridad referencial.")

except Error as e:
    print(f"Error al conectar o ejecutar comandos en MySQL: {e}")

finally:
    # 8. Cierre de cursores y conexión
    if 'conexion' in locals() and conexion.is_connected():
        cursor.close()
        conexion.close()
        print("Conexión a MySQL cerrada.")

# **INSERCION DE LOS DATOS DESDE PYTHON**

# Orden de Inserción e Integridad Referencial

En el diseño y gestión de bases de datos relacionales existe un principio fundamental: la **integridad referencial**. Esta regla establece que una clave foránea (`FOREIGN KEY`) siempre debe apuntar a un registro que ya exista previamente en la tabla de origen (`PRIMARY KEY`).

---

## 1. La Regla del "Padre antes que el Hijo"

En un modelo relacional, las tablas se clasifican según sus dependencias:

* **Tablas Padres (Independientes):** No requieren de otras tablas para existir.
* **Tablas Hijas (Dependientes):** Contienen claves foráneas que referencian a otras tablas.

> **Principio clave:** No se puede insertar un registro en una tabla hija si el identificador al que hace referencia aún no existe en la tabla padre. Intentar hacerlo generará un error de violación de restricción de clave foránea (`Foreign Key Constraint Violation`).

---

## 2. Árbol de Dependencias del Sistema

Al examinar las relaciones de la base de datos de exoplanetas:

```text
  [ ESTRELLA ]       [ TELESCOPIO ]   <--- Tablas Padres (Independientes)
       │                   │
       ▼                   │
   [ PLANETA ]             │          <--- Tabla Hija de Estrella / Padre de Descubrimiento
       │                   │
       └─────────┬─────────┘
                 ▼
          [ DESCUBRIMIENTO ]          <--- Tabla Hija dependiente de Planeta y Telescopio

```

* **`estrella`**: Es totalmente independiente (Padre).
* **`telescopio`**: Es totalmente independiente (Padre).
* **`planeta`**: Depende directamente de `estrella` a través de `id_estrella`.
* **`descubrimiento`**: Depende simultáneamente de `planeta` (`id_planeta`) y `telescopio` (`id_telescopio`).

---

## 3. Orden Lógico de Ejecución (`INSERT`)

Siguiendo la jerarquía de dependencias, el orden estricto para poblar las tablas es:

1. **`estrella`** → Debe registrarse primero la estrella hospedadora (ej. *Kepler-186*).
2. **`telescopio`** → Debe registrarse el instrumento u observatorio (ej. *Kepler*). *(Los pasos 1 y 2 son intercambiables entre sí)*.
3. **`planeta`** → Requiere que el `id_estrella` asociado ya exista en la base de datos.
4. **`descubrimiento`** → Requiere que tanto el `id_planeta` como el `id_telescopio` existan previamente.

---

### Resumen

> Para registrar que el telescopio **Kepler** descubrió el exoplaneta **Kepler-186f** alrededor de la estrella **Kepler-186**, la base de datos necesita validar primero la existencia de la estrella, asociar el planeta a dicha estrella, registrar el telescopio utilizado y, finalmente, vincular ambos identificadores en la tabla de descubrimientos.

In [ ]:
import mysql.connector

In [ ]:
conexion = mysql.connector.connect(
    host="localhost",
    user="root",
    password="1234",
    database="astronomia"
)

cursor = conexion.cursor()